In [1]:
import os
OPENAI_API_KEY = os.environ['OPENAI_API_KEY']

from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings

In [2]:
cornwall_granular_collection = Chroma(
    collection_name="cornwall_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)

In [3]:
cornwall_granular_collection.reset_collection()

In [4]:
cornwall_coarse_collection = Chroma(
    collection_name="cornwall_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY)
)
cornwall_granular_collection.reset_collection()

In [5]:
os.environ["USER_AGENT"] = "manning-ch08/1.0 (mic.a.elle.chlon@gmail.com)"

from langchain_community.document_loaders import AsyncHtmlLoader
destination_url = "https://en.wikivoyage.org/wiki/Cornwall"
html_loader = AsyncHtmlLoader(destination_url)
docs = html_loader.load()

Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.18it/s]


In [6]:
from langchain_text_splitters import HTMLSectionSplitter

header_to_split_on = [("h1", "Header 1"), ("h2", "Header 2")]
html_section_splitter = HTMLSectionSplitter(
    headers_to_split_on=header_to_split_on
)

def split_docs_into_granular_chunks(docs):
    all_chunks = []
    for doc in docs:
        html_string = doc.page_content
        temp_chunks = html_section_splitter.split_text(
            html_string
        )
        all_chunks.extend(temp_chunks)

    return all_chunks

In [7]:
granular_chunks = split_docs_into_granular_chunks(docs)

cornwall_granular_collection.add_documents(documents=granular_chunks)

['345893a1-97f5-40c1-9e8b-73048096ee93',
 '77733009-64e9-48e6-8467-2a5842181769',
 '9370d2a7-f0c1-438c-9990-67065ce26d3d',
 'd0f8c700-4f6a-44d0-b2c9-588c365bbb16',
 '61d54335-51e6-4bc2-b6b6-1400156310e4',
 '72e7c0ba-d387-44ce-b69b-8f56d02e8676',
 'bb14e024-ffbc-4d7f-ba92-a2b4ded51eef',
 'e2f29e88-b108-4a83-9917-7806dd8759a2',
 'd750163a-ca84-4c5d-96a8-447ef875c230',
 'eee3c413-fb24-4fcf-844f-844e59db8030',
 '488728ad-d1a4-4496-8559-f4c2935e2c7b',
 'cbdb0203-9e07-4bdf-9307-c641cdc148e7',
 'b6275763-ff5b-413a-9c6b-acc0af9596bf',
 '16f818fa-f725-491e-9400-bc4512d693e4',
 'f2911230-df05-408a-8047-f100bed4fa88',
 '56f4fb45-1443-452f-9a44-a67fbf123821',
 '17dd4a7a-e70d-464d-8624-88e2ddbf0fc1',
 'd5c17148-29b9-45ec-867b-5a9108738d56',
 'd7c7319c-8543-4a33-a08c-7286758d9626']

In [8]:
results = cornwall_granular_collection.similarity_search(
    query="Events or festivals in Cornwall",k=3
)

for doc in results:
    print(doc)

page_content='Cornwall' metadata={'Header 1': 'Cornwall'}
page_content='Festivals 
 [ edit ] 
 
 These festivals tend to not be public holidays and not all are celebrated fully across the county. 
   
 AberFest .   A Celtic cultural festival celebrating “All things” Cornish and Breton that takes place biennially (every two years) in Cornwall at Easter. The AberFest Festival alternates with the Breizh – Kernow Festival that is held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate years.       ( updated Jun 2023 ) 
 Golowan , sometimes also  Goluan  or  Gol-Jowan  is the Cornish word for the Midsummer celebrations, most popular in the Penwith area and in particular  Penzance  and  Newlyn . The celebrations are conducted from the 23rd of June (St John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve being the more popular in Cornish fishing communities. The celebrations are centred around the lighting of bonfires and fireworks and the performance 

In [9]:
from langchain_community.document_transformers import Html2TextTransformer
from langchain_text_splitters import RecursiveCharacterTextSplitter

html2text_transformer = Html2TextTransformer()
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=3000, chunk_overlap=300
)

In [10]:
def split_docs_into_coarse_chunks(docs):
    text_docs = html2text_transformer.transform_documents(docs)
    coarse_chunks = text_splitter.split_documents(text_docs)
    return coarse_chunks

In [11]:
coarse_chunks = split_docs_into_coarse_chunks(docs)

cornwall_coarse_collection.add_documents(documents=coarse_chunks)

['415e8260-032b-46aa-8121-0c248803b4fc',
 '8b02f41f-2da6-4ddd-bc41-642b53524520',
 '18fac7bd-3887-4425-9b51-3789cb90b74e',
 '4b78404c-3d37-4cf3-8662-d319af6ad203',
 '8dbdcaad-1940-4bd6-904c-bf2b352d57b9',
 'abf649d9-1e1b-41f7-83d8-9e165a56ac36',
 'd1a0ded0-b6fc-4700-9168-4257c92f7a32',
 '32c0afaa-edb6-4366-868c-9b556d5c3518',
 '278570ec-0e3e-485c-9244-830726a410f4',
 'd7e9c866-f1ab-4d3f-9b4f-16a8b266b2d9',
 '3ccb994f-6b30-4db0-8f05-3f5482b14ed1',
 '4a4c964a-266e-4c70-b26f-c85c42ab2e55',
 '7da440b3-9dc8-4ee2-95af-a42503afb593',
 'c40c2def-38d9-42ab-a9a7-83ba5aafe59e',
 '03a70515-d576-4843-9155-ce5a34a8ea70']

In [12]:
results = cornwall_coarse_collection.similarity_search(
    query="Events or Festival in Cornwall", k=3
)

for doc in results:
    print(doc)

page_content='### Spirits

[edit]

    _See also:Liquor_

Gin and rum are also produced in Cornwall. A popular brand of Cornish rum is
Dead Man's Fingers which has multiple flavours and is bottled in St. Ives.

## Festivals

[edit]

These festivals tend to not be public holidays and not all are celebrated
fully across the county.

AberFest. A Celtic cultural festival celebrating “All things” Cornish and
Breton that takes place biennially (every two years) in Cornwall at Easter.
The AberFest Festival alternates with the Breizh – Kernow Festival that is
held in Brandivy and Bignan (in Breizh/Bretagne – France) on the alternate
years. (updated Jun 2023)

**Golowan** , sometimes also _Goluan_ or _Gol-Jowan_ is the Cornish word for
the Midsummer celebrations, most popular in the Penwith area and in particular
Penzance and Newlyn. The celebrations are conducted from the 23rd of June (St
John's Eve) to the 28th of June (St Peter's Eve) each year, St Peter's Eve
being the more popular in Corni

In [13]:
uk_granular_collection = Chroma(
    collection_name="uk_granular",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_granular_collection.reset_collection()

uk_coarse_collection = Chroma(
    collection_name="uk_coarse",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY),
)
uk_coarse_collection.reset_collection()

uk_destinations = [
    "Cornwall", "North_Cornwall", "South_Cornwall", "West_Cornwall",
    "Tintagel", "Bodmin", "Wadebridge", "Penzance", "Newquay",
    "St_Ives", "Port_Isaac", "Looe", "Polperro", "Porthleven",
    "East_Sussex", "Brighton", "Battle", "Hastings_(England)",
    "Rye_(England)", "Seaford", "Ashdown_Forest"
]

wikivoyage_root_url = "https://en.wikivoyage.org/wiki"
uk_destination_urls = [f'{wikivoyage_root_url}/{d}' 
                       for d in uk_destinations]

for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    docs = html_loader.load()

granular_chunks = split_docs_into_granular_chunks(docs)
uk_granular_collection.add_documents(documents=granular_chunks)

coarse_chunks = split_docs_into_coarse_chunks(docs)
uk_coarse_collection.add_documents(documents=coarse_chunks)

Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  7.14it/s]


['0f6876ec-561a-42cb-9ebe-b9f4398fd95a',
 '05ef1ff0-426b-41bd-8b30-9a02154c9efc',
 '46b2001c-4096-42b0-8878-2c54dbde0a2d',
 '9649b6b0-20b0-4df2-8c35-84d4ec1501a9',
 'cda4d04e-2102-4f3e-99ac-e717b749421c',
 '2cf3d5df-6308-4432-af86-f053ba762eec',
 '95408bd3-df5c-4f24-b13f-e03da5ae82d0',
 'd36e100f-7684-43d1-a597-3424ffa669e3',
 '0cee1a6f-f14e-4ff5-a12a-d0c5734bd5d0']

In [14]:
granular_results = uk_granular_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in granular_results:
    print(doc)
    print("\n------------------------------------------------------------------\n")

coarse_results = uk_coarse_collection.similarity_search(
    query="Events or festivals in East Sussex",k=4)

for doc in coarse_results:
    print(doc)

page_content='Go next 
 [ edit ] 
 
 
 
 Royal Tunbridge Wells  (on the A26) - Victorian spa town with bars, pubs and drinking fountains for the local water. 
 Eastbourne 
 Petersfield 
 
 South Downs Way , a popular walking path. 
 
 London , a train ride away. 
 Kent 
 Medway 
 Crowborough 
 
 
 
 
 Routes through Ashdown Forest 
 
 
 
 
 
 
 
 
 London   ←   East Grinstead   ← 
 
 
   N     S   
 
 
 →   Uckfield   →   Eastbourne 
 
 
 
 
 .mw-parser-output .routeBox{font-size:small;border-style:none;border-spacing:0 0;border-collapse:collapse;margin:0 auto}.mw-parser-output .routeBox td{padding:1px 2px} 
 
 
 
 
 
 
 
 .mw-parser-output .article-status{width:60%;background:#fff;color:black;margin:0 auto;border:solid 2px lightblue;text-align:center;font-size:90%;font-style:italic}.mw-parser-output .article-status-disambig{border:2px dashed lightblue}.mw-parser-output .article-status-disambig td:first-child{width:48px;text-align:center}.mw-parser-output .article-status-stub{border:1p

## 8.4/ Embedding strategy

In [15]:
from langchain_classic.retrievers import ParentDocumentRetriever
from langchain_classic.storage import InMemoryStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [16]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunk_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
)
child_chunk_collection.reset_collection()

doc_store = InMemoryStore()

parent_doc_retriever = ParentDocumentRetriever(
    vectorstore=child_chunk_collection,
    docstore=doc_store,
    child_splitter=child_splitter,
    parent_splitter=parent_splitter
)

In [17]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)
    print(f'Ingesting {destination_url}')
    parent_doc_retriever.add_documents(text_docs, ids=None)

Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.60it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.01it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.06it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.23it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  5.46it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [18]:
list(doc_store.yield_keys())

['a81a443b-7253-46bb-a93f-ccca4be908ee',
 'a8a4ad93-2677-4f04-8ad1-90504c3c6311',
 'c6d46338-5155-49df-b768-4482d9297355',
 '1ab1a3f9-ca2f-4c18-8539-b0220ad7403c',
 'a22dba51-1fde-4643-b326-e455bc748da3',
 'b9a7b8e0-5f55-4a23-9d18-b5a8a5dda1bd',
 'dcb2807c-cf78-4fb7-8eaa-9b86f98f5186',
 'a750e2a6-268d-49bf-87cf-5a853079631b',
 'ccc23e08-c718-451e-ab49-2e36c8734c7a',
 '55b872d5-b47c-47b7-9e29-ade07fbca0f4',
 'c262886b-bd19-4e1c-857d-6f098b6b34ba',
 'af4056d9-09b0-4ecb-8a70-eee2c8cedf85',
 '660e8dcf-9960-435b-a363-115a9e7ebc52',
 'b79642ac-d720-48d7-acd4-522857626a84',
 'ad67e3ac-e714-4602-a781-e7d79f81ab6b',
 '1ee14281-e946-44d5-b5b0-09515888d3aa',
 'de132c4e-8705-4de4-a460-38e9f487b219',
 'fea521ce-979b-4132-910c-e954296f8ae5',
 '3b259ec6-7a48-489c-aa6e-3fb83118174d',
 '46a706aa-2a95-4dd0-ac3a-b7861959f278',
 'e43185a8-121a-43ab-92b6-bbb9a2f9b7ee',
 '33afdf79-a97d-4b40-ab50-12176fff8865',
 '3afd5a09-7990-4920-850a-1c591b6be735',
 'c6fbf41e-08af-425e-909f-20c398819515',
 '7773fec3-b966-

In [19]:
retrieved_docs = parent_doc_retriever.invoke("Cornwall Ranger")

In [20]:
print(retrieved_docs[0])

page_content='## Get around

[edit]

### By bus

[edit]

Thanks to Transport for Cornwall, all bus tickets are interchangeable across
the different companies. The **Cornwall All Day ticket** allows unlimited
travel for a calendar day. As of 2023, fares are £5 for adults and £4 for
under-19s. Payment is by cash or contactless. The two main bus companies are:

  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).
  * **Kernow** (part of First Bus) covers western and central Cornwall.

Buses only serve designated stops when in towns; otherwise, you can flag them
down anywhere that's safe for them to stop.

### By train

[edit]

**CrossCountry Trains** and **Great Western Railway** operate regular train
services between the main centres of population, the latter company also
serving a number of other towns on branch lines. For train times and fares
visit National Rail Enquiries.

The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall 

In [21]:
child_docs_only = child_chunk_collection.similarity_search("Cornwall Ranger")

In [22]:
print(child_docs_only)

[Document(id='8f602f37-2b90-4c5f-8ec9-745e8b064655', metadata={'language': 'en', 'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'doc_id': 'c6fbf41e-08af-425e-909f-20c398819515', 'title': 'South Cornwall – Travel guide at Wikivoyage'}, page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and\nPlymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for\nunder-16s.\n\n## See\n\n[edit]\n\nThe **Eden Project** , near St Austell, a fabulous collection of flora from\nall over the planet housed in two space age transparent domes, and a massive\nzip line.'), Document(id='bdba9735-e327-452d-a526-2af45d96e3ab', metadata={'title': 'Cornwall – Travel guide at Wikivoyage', 'doc_id': '1ab1a3f9-ca2f-4c18-8539-b0220ad7403c', 'source': 'https://en.wikivoyage.org/wiki/Cornwall', 'language': 'en'}, page_content='### Cornish\n\n[edit]'), Document(id='3528c0e4-572a-478c-ba09-09af657e70f9', metadata={'title': 'Cornwall – Travel guide at Wikivoya

### 8.4.2/ Embedding child chunks with MultiVectorRetriever

In [23]:
from langchain_classic.retrievers.multi_vector import MultiVectorRetriever
from langchain_classic.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import AsyncHtmlLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
import uuid

In [24]:
parent_splitter = RecursiveCharacterTextSplitter(chunk_size=3000)
child_splitter = RecursiveCharacterTextSplitter(chunk_size=500)

child_chunks_collection = Chroma(
    collection_name="uk_child_chunks",
    embedding_function=OpenAIEmbeddings(api_key=OPENAI_API_KEY) 
)
child_chunks_collection.reset_collection()

doc_byte_store = InMemoryByteStore()
doc_key = "doc_id"

multi_vector_retriever = MultiVectorRetriever(
    vectorstore=child_chunks_collection,
    byte_store=doc_byte_store
)

In [25]:
for destination_url in uk_destination_urls:
    html_loader = AsyncHtmlLoader(destination_url)
    html_docs = html_loader.load()
    text_docs = html2text_transformer.transform_documents(html_docs)

    coarse_chunks = parent_splitter.split_documents(text_docs)
    
    coarse_chunks_ids = [str(uuid.uuid4()) for _ in coarse_chunks]
    
    all_granular_chunks = []
    for i, coarse_chunk in enumerate(coarse_chunks):

        coarse_chunk_id = coarse_chunks_ids[i]
        granular_chunks = child_splitter.split_documents([coarse_chunk])

        for granular_chunk in granular_chunks:
            granular_chunk.metadata[doc_key] = coarse_chunk_id

        all_granular_chunks.extend(granular_chunks)
    
    print(f'Ingesting {destination_url}')
    multi_vector_retriever.vectorstore.add_documents(all_granular_chunks)
    multi_vector_retriever.docstore.mset(
        list(zip(coarse_chunks_ids, coarse_chunks)))

Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.43it/s]


Ingesting https://en.wikivoyage.org/wiki/Cornwall


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  7.48it/s]


Ingesting https://en.wikivoyage.org/wiki/North_Cornwall


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.16it/s]


...

Ingesting https://en.wikivoyage.org/wiki/Rye_(England)


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  7.31it/s]


Ingesting https://en.wikivoyage.org/wiki/Seaford


Fetching pages: 100%|#############################| 1/1 [00:00<00:00,  6.70it/s]


Ingesting https://en.wikivoyage.org/wiki/Ashdown_Forest


In [26]:
retrieved_docs = multi_vector_retriever.invoke("Cornwall Ranger")
print(retrieved_docs)

[Document(metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}, page_content="## Get around\n\n[edit]\n\n### By bus\n\n[edit]\n\nThanks to Transport for Cornwall, all bus tickets are interchangeable across\nthe different companies. The **Cornwall All Day ticket** allows unlimited\ntravel for a calendar day. As of 2023, fares are £5 for adults and £4 for\nunder-19s. Payment is by cash or contactless. The two main bus companies are:\n\n  * **Go Cornwall Bus** covers all parts of Cornwall and connects with Plymouth (in Devon).\n  * **Kernow** (part of First Bus) covers western and central Cornwall.\n\nBuses only serve designated stops when in towns; otherwise, you can flag them\ndown anywhere that's safe for them to stop.\n\n### By train\n\n[edit]\n\n**CrossCountry Trains** and **Great Western Railway** operate regular train\nservices between the main centres of population, the latter company also\ns

In [28]:
child_docs_only = child_chunks_collection.similarity_search("Cornwall Ranger")
print(child_docs_only[0])

page_content='The **Cornwall Ranger** ticket allows unlimited train travel in Cornwall and
Plymouth for a calendar day. As of 2023, this costs £14 for adults and £7 for
under-16s.

## See

[edit]

The **Eden Project** , near St Austell, a fabulous collection of flora from
all over the planet housed in two space age transparent domes, and a massive
zip line.' metadata={'source': 'https://en.wikivoyage.org/wiki/South_Cornwall', 'doc_id': '625b88af-fa85-4cd1-acfa-42d830c4482b', 'title': 'South Cornwall – Travel guide at Wikivoyage', 'language': 'en'}


### 8.4.3/ Embedding document summaries 